In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

In [2]:
df = pd.read_csv("train (1).csv")
# Target: log-transform to reduce skew
df["SalePrice_log"] = np.log1p(df["SalePrice"])

# Categorical NA means "Feature absent" for these columns
non_cols = [
    "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu", "GarageType", "GarageFinish", "GarageQual", 
    "GarageCond", "BsmtQual", "BsmtCond", "BsmtExposure"
]

for c in non_cols:
  df[c] = df[c].fillna("None")

num_cols = df.select_dtypes(include=np.number).columns.drop(["SalePrice", "SalePrice_log", "Id"])

df[num_cols] = df[num_cols].fillna(df[num_cols].median())

df = pd.get_dummies(df.select_dtypes(exclude="object").join(pd.get_dummies(df.select_dtypes(include="object"),drop_first=True)))


In [4]:
X = df.drop(columns=["SalePrice", "SalePrice_log", "Id"])
y = df["SalePrice_log"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

for name, model in [("Ridge", Ridge(alpha=10)), ("Lasso", Lasso(alpha=0.001))]:
    model.fit(X_train_s, y_train)
    preds = model.predict(X_test_s)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    print(f"{name} log-RMSE: {rmse:.4f}")

Ridge log-RMSE: 0.1559
Lasso log-RMSE: 0.1556


c:\Users\Mahyar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.746e-02, tolerance: 1.781e-02
  model = cd_fast.enet_coordinate_descent(
